# Sudoku RL with TRL GRPO

This notebook creates a separate GRPO-based training workflow for the Sudoku project using `trl`.

The policy is trained to emit exactly one JSON move of the form `{"row": 1-9, "column": 1-9, "value": 1-9}` for a partially filled Sudoku board. Rewards are computed from Sudoku rules and the hidden solution board, which keeps the loop lightweight and notebook-friendly.

In [ ]:
import sys
from pathlib import Path

print("Python executable:", sys.executable)
print("Working directory:", Path.cwd())

try:
    import trl  # noqa: F401
    import datasets  # noqa: F401
    import peft  # noqa: F401
    print("trl, datasets, and peft are already available in this kernel.")
except ModuleNotFoundError:
    print("One or more training dependencies are missing in this notebook kernel.")
    print("Run the next install cell with %pip, then restart the kernel.")

In [ ]:
# %pip install -q trl datasets peft accelerate transformers

In [ ]:
import json
import os
import random
import re
import textwrap
from typing import Any

import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

from sudoku_rl.sudoku_logic import build_initial_puzzle, format_board_text, generate_valid_sudoku

MODEL_NAME = os.getenv("MODEL_NAME", "Qwen/Qwen2.5-0.5B-Instruct")
OUTPUT_DIR = os.getenv("OUTPUT_DIR", "grpo-sudoku-checkpoints")
SEED = int(os.getenv("SEED", "7"))
TRAIN_EXAMPLES = int(os.getenv("TRAIN_EXAMPLES", "256"))
EVAL_EXAMPLES = int(os.getenv("EVAL_EXAMPLES", "8"))
MIN_HOLES = int(os.getenv("MIN_HOLES", "25"))
MAX_HOLES = int(os.getenv("MAX_HOLES", "45"))

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

SYSTEM_PROMPT = textwrap.dedent(
    """
    You are solving a Sudoku puzzle one move at a time.

    Return exactly one JSON object with this schema:
    {"row": <1-9>, "column": <1-9>, "value": <1-9>}

    Rules:
    - Choose one editable empty cell.
    - The value must obey Sudoku row, column, and 3x3 box constraints.
    - Prefer a move that is correct in the hidden final solution.
    - Do not include markdown, explanations, or code fences.
    """
).strip()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model:", MODEL_NAME)
print("CUDA available:", torch.cuda.is_available())
print("bf16 supported:", torch.cuda.is_available() and torch.cuda.is_bf16_supported())

In [ ]:
def serialize_grid(grid: list[list[int]]) -> str:
    return json.dumps(grid, separators=(",", ":"))


def make_prompt(puzzle: list[list[int]]) -> str:
    board_text = format_board_text(puzzle)
    return textwrap.dedent(
        f"""
        {SYSTEM_PROMPT}

        Current puzzle:
        {board_text}

        Return the next move as JSON only.
        """
    ).strip()


def make_example(example_seed: int, holes: int) -> dict[str, Any]:
    rng = random.Random(example_seed)
    solution = generate_valid_sudoku(rng)
    puzzle = build_initial_puzzle(solution, holes=holes, rng=rng)
    return {
        "prompt": make_prompt(puzzle),
        "puzzle": serialize_grid(puzzle),
        "solution": serialize_grid(solution),
        "holes": holes,
    }


def build_dataset(num_examples: int, seed: int = SEED) -> Dataset:
    rows: list[dict[str, Any]] = []
    rng = random.Random(seed)
    for offset in range(num_examples):
        holes = rng.randint(MIN_HOLES, MAX_HOLES)
        rows.append(make_example(example_seed=seed + offset, holes=holes))
    return Dataset.from_list(rows)


train_dataset = build_dataset(TRAIN_EXAMPLES, seed=SEED)
eval_dataset = build_dataset(EVAL_EXAMPLES, seed=SEED + 10_000)

train_dataset[0]

In [ ]:
ACTION_RE = re.compile(r"\{[^{}]+\}", flags=re.DOTALL)


def load_grid(payload: str | list[list[int]]) -> list[list[int]]:
    if isinstance(payload, str):
        return json.loads(payload)
    return payload


def extract_completion_text(completion: Any) -> str:
    if isinstance(completion, str):
        return completion.strip()

    if isinstance(completion, dict):
        content = completion.get("content", "")
        if isinstance(content, str):
            return content.strip()
        if isinstance(content, list):
            return "".join(part.get("text", "") for part in content if isinstance(part, dict)).strip()

    if isinstance(completion, list):
        parts: list[str] = []
        for item in completion:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                content = item.get("content", "")
                if isinstance(content, str):
                    parts.append(content)
                elif isinstance(content, list):
                    parts.extend(part.get("text", "") for part in content if isinstance(part, dict))
        return "".join(parts).strip()

    return str(completion).strip()


def parse_action(text: str) -> dict[str, int] | None:
    cleaned = (text or "").strip()
    if not cleaned:
        return None

    candidates: list[dict[str, Any]] = []

    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, dict):
            candidates.append(parsed)
    except json.JSONDecodeError:
        pass

    for match in ACTION_RE.finditer(cleaned):
        try:
            parsed = json.loads(match.group(0))
        except json.JSONDecodeError:
            continue
        if isinstance(parsed, dict):
            candidates.append(parsed)

    row_match = re.search(r'"?row"?\s*[:=]\s*([1-9])', cleaned, flags=re.IGNORECASE)
    column_match = re.search(r'"?(?:column|col)"?\s*[:=]\s*([1-9])', cleaned, flags=re.IGNORECASE)
    value_match = re.search(r'"?(?:value|number)"?\s*[:=]\s*([1-9])', cleaned, flags=re.IGNORECASE)
    if row_match and column_match and value_match:
        candidates.append(
            {
                "row": int(row_match.group(1)),
                "column": int(column_match.group(1)),
                "value": int(value_match.group(1)),
            }
        )

    for candidate in candidates:
        try:
            row = int(candidate["row"])
            column = int(candidate["column"])
            value = int(candidate.get("value", candidate.get("number")))
        except (KeyError, TypeError, ValueError):
            continue
        if 1 <= row <= 9 and 1 <= column <= 9 and 1 <= value <= 9:
            return {"row": row, "column": column, "value": value}

    return None


def legal_move(puzzle: list[list[int]], row: int, column: int, value: int) -> bool:
    row_index = row - 1
    column_index = column - 1

    if puzzle[row_index][column_index] != 0:
        return False

    if value in puzzle[row_index]:
        return False

    if value in [puzzle[r][column_index] for r in range(9)]:
        return False

    box_row = (row_index // 3) * 3
    box_column = (column_index // 3) * 3
    for sub_row in range(box_row, box_row + 3):
        for sub_column in range(box_column, box_column + 3):
            if puzzle[sub_row][sub_column] == value:
                return False

    return True


In [ ]:
def format_reward(completions: list[Any], **_: Any) -> list[float]:
    rewards: list[float] = []
    for completion in completions:
        text = extract_completion_text(completion)
        action = parse_action(text)
        rewards.append(0.25 if action is not None else 0.0)
    return rewards


def legality_reward(completions: list[Any], puzzle: list[str], **_: Any) -> list[float]:
    rewards: list[float] = []
    for completion, puzzle_payload in zip(completions, puzzle):
        board = load_grid(puzzle_payload)
        action = parse_action(extract_completion_text(completion))
        if action is None:
            rewards.append(0.0)
            continue

        editable_reward = 0.25 if board[action["row"] - 1][action["column"] - 1] == 0 else 0.0
        sudoku_reward = 0.75 if legal_move(board, action["row"], action["column"], action["value"]) else 0.0
        rewards.append(editable_reward + sudoku_reward)
    return rewards


def correctness_reward(completions: list[Any], puzzle: list[str], solution: list[str], **_: Any) -> list[float]:
    rewards: list[float] = []
    for completion, puzzle_payload, solution_payload in zip(completions, puzzle, solution):
        board = load_grid(puzzle_payload)
        solution_board = load_grid(solution_payload)
        action = parse_action(extract_completion_text(completion))
        if action is None:
            rewards.append(0.0)
            continue

        row_index = action["row"] - 1
        column_index = action["column"] - 1
        if board[row_index][column_index] != 0:
            rewards.append(0.0)
            continue

        rewards.append(2.0 if solution_board[row_index][column_index] == action["value"] else 0.0)
    return rewards


def completion_reward(completions: list[Any], puzzle: list[str], solution: list[str], **_: Any) -> list[float]:
    rewards: list[float] = []
    for completion, puzzle_payload, solution_payload in zip(completions, puzzle, solution):
        board = load_grid(puzzle_payload)
        solution_board = load_grid(solution_payload)
        action = parse_action(extract_completion_text(completion))
        if action is None or not legal_move(board, action["row"], action["column"], action["value"]):
            rewards.append(0.0)
            continue

        row_index = action["row"] - 1
        column_index = action["column"] - 1
        remaining_empty = sum(1 for row in board for value in row if value == 0)
        solved_fraction = 1.0 - ((remaining_empty - 1) / max(remaining_empty, 1))
        if solution_board[row_index][column_index] == action["value"]:
            rewards.append(0.5 + solved_fraction)
        else:
            rewards.append(0.0)
    return rewards


## GRPO trainer setup

The LoRA target modules below are set up for Qwen-style decoder blocks. If you switch to a very different base model, update `target_modules` to match that architecture.

In [ ]:
bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16 = torch.cuda.is_available() and not bf16

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    num_generations=4,
    max_prompt_length=768,
    max_completion_length=96,
    logging_steps=1,
    save_steps=50,
    save_strategy="steps",
    report_to="none",
    remove_unused_columns=False,
    bf16=bf16,
    fp16=fp16,
)

trainer = GRPOTrainer(
    model=MODEL_NAME,
    reward_funcs=[
        format_reward,
        legality_reward,
        correctness_reward,
        completion_reward,
    ],
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config,
)

trainer

In [ ]:
train_result = trainer.train()
train_result

In [ ]:
def generate_move(prompt: str, max_new_tokens: int = 64) -> str:
    model = trainer.model
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


for sample in eval_dataset.select(range(min(3, len(eval_dataset)))):
    completion = generate_move(sample["prompt"])
    action = parse_action(completion)
    print("Board:")
    print(format_board_text(load_grid(sample["puzzle"])))
    print("Model completion:", completion)
    print("Parsed action:", action)
    print()


In [ ]:
final_dir = os.path.join(OUTPUT_DIR, "final")
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print("Saved GRPO checkpoint to", final_dir)